In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window


In [0]:
spark.read.format('csv').option('header',True).option('inferSchema',True).load('/Volumes/namastespark/namasteschema/vols/bronze/20260209/online_customers.csv').write.format('delta').mode('append').saveAsTable('online_customers')

In [0]:
spark.read.format('csv').option('header',True).option('inferSchema',True).load('/Volumes/namastespark/namasteschema/vols/bronze/20260209/offline_customers.csv').write.format('delta').mode('append').saveAsTable('offline_customers')

In [0]:
spark.read.format('csv').option('header',True).option('inferSchema',True).load('/Volumes/namastespark/namasteschema/vols/bronze/20260209/offline_orders.csv').write.format('delta').mode('append').saveAsTable('offline_orders')

In [0]:
spark.read.format('csv').option('header',True).option('inferSchema',True).load('/Volumes/namastespark/namasteschema/vols/bronze/20260209/online_orders.csv').write.format('delta').mode('append').saveAsTable('online_orders')

In [0]:
spark.read.format('csv').option('header',True).option('inferSchema',True).load('/Volumes/namastespark/namasteschema/vols/bronze/20260209/offline_products.csv').write.format('delta').mode('append').saveAsTable('offline_products')

In [0]:
spark.read.format('csv').option('header',True).option('inferSchema',True).load('/Volumes/namastespark/namasteschema/vols/bronze/20260209/online_products.csv').write.format('delta').mode('append').saveAsTable('online_products')

In [0]:
# product_df= spark.sql('''
# SELECT
#   op.sku_id,
#   LOWER(op.product_name) AS product_name,
#   LOWER(op.category)     AS category,
#   LOWER(op.brand)        AS brand,
#   ROUND(op.price, 2)     AS price,
#   op.currency,
#   CAST(op.is_active AS BOOLEAN) AS is_active,
#   ROUND(ofp.mrp_price, 2)       AS mrp_price,
#   ROUND(ofp.tax_rate * 100, 2)  AS tax_rate, -- now as percentage with 2 decimals
#  current_date() as ingested_date, year(current_date) as year, month(current_date) as month , day(current_date) as day 
# FROM online_products op
# INNER JOIN offline_products ofp
#   ON op.sku_id = ofp.sku;  -- change to ofp.sku if that's your actual column
#   ''')

# product_df.display()
# #product_df.write.format('delta').partitionBy('year','month','day').mode('append').saveAsTable('products_cleansed')

In [0]:
# %sql
# WITH cte1 AS (
#   SELECT
#     customer_id,
#     TRIM(full_name) AS full_name,
#     LOWER(email) AS email,
#     RIGHT(mobile_number, 10) AS mobile_no,
#     city,
#     state,
#     zipcode,
#     signup_date
#   FROM online_customers
# ),
# cte2 AS (
#   SELECT
#     cust_id AS customer_id,
#     name,
#     LOWER(email) AS email,
#     RIGHT(phone, 10) AS mobile_no,
#     city,
#     state,
#     pincode AS zipcode,
#     store_id
#   FROM offline_customers
# )
# SELECT
#   c1.*,
#   c2.store_id,
#   CURRENT_DATE() AS ingested_date,
#   YEAR(CURRENT_DATE) AS year,
#   MONTH(CURRENT_DATE) AS month,
#   DAY(CURRENT_DATE) AS day
# FROM cte1 AS c1
# LEFT JOIN cte2 AS c2
#   ON c1.mobile_no = c2.mobile_no


In [0]:
# %sql
# WITH cte1 AS (
#   SELECT
#     order_id,
#     DATE_FORMAT(order_ts, 'yyyy-MM-dd') AS order_date,
#     customer_id,
#     sku_id,
#     qty,
#     list_price AS sale_price,
#     discount,
#     CURRENT_DATE() AS ingested_date,
#     YEAR(CURRENT_DATE) AS year,
#     MONTH(CURRENT_DATE) AS month,
#     DAY(CURRENT_DATE) AS day
#   FROM online_orders
# ),
# cte2 as ( 
# SELECT 
#    LOWER(txn_id) AS order_id,
#    DATE_FORMAT(txn_time, 'yyyy-MM-dd') AS order_date,
#   customer_id , sku, quantity as qty, round(mrp,2) as sale_price, discount , payment_mode,store_num as store_id,
#    CURRENT_DATE() AS ingested_date,
#    YEAR(CURRENT_DATE) AS year,
#    MONTH(CURRENT_DATE) AS month,
#    DAY(CURRENT_DATE) AS day
#    FROM offline_orders
# )
# select c1.* ,c2.payment_mode,  c2.store_id
# from cte1 as c1 left join cte2 as c2 
# on c1.customer_id = c2.customer_id
